#LSTM-GRU + RF Hybrid Model

*   Pre-trained embeddings: Uses GloVe 100D instead of random embeddings.
*   Bidirectional GRU: More stable & faster than LSTM.
*   Feature selection for Random Forest (RF): Uses TF-IDF instead of raw sequences.
*   Hyperparameter tuning: Uses RandomizedSearchCV to optimize RF hyperparameters.

In [ ]:
import pandas as pd
import numpy as np
import re
import tensorflow as tf
import gensim.downloader as api

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, GRU, Dense, Dropout, Bidirectional, GlobalAveragePooling1D
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score

In [ ]:
# Load Dataset
df = pd.read_csv('/content/sample_data/Tweets.csv', encoding='utf-8')[['airline_sentiment', 'text']]
df.dropna(inplace=True)

In [ ]:
# Clean text
def clean_text(text):
    text = re.sub(r'http\S+|www\S+', '', text)  # Remove URLs
    text = re.sub(r'[^a-zA-Z\s]', '', text)  # Remove special characters
    return text.lower().strip()

df['text'] = df['text'].astype(str).apply(clean_text)

# Encode labels
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df['airline_sentiment'])

# Tokenization
VOCAB_SIZE = 20000  # Increased vocab size
MAX_LEN = 100

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(df['text'])
sequences = tokenizer.texts_to_sequences(df['text'])
X = pad_sequences(sequences, maxlen=MAX_LEN, padding='post', truncating='post')

# Load GloVe embeddings (100D)
embedding_dim = 100
word_vectors = api.load("glove-twitter-100")

# Create embedding matrix
embedding_matrix = np.zeros((VOCAB_SIZE, embedding_dim))
for word, i in tokenizer.word_index.items():
    if i < VOCAB_SIZE and word in word_vectors:
        embedding_matrix[i] = word_vectors[word]


[==================================================] 100.0% 387.1/387.1MB downloaded


**Split Dataset**

In [ ]:
# Split dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

**The Model: LSTM-GRU and RF with Hypertuning**

In [ ]:
from tensorflow.keras import Model, Input

# Define a functional model instead of Sequential
input_layer = Input(shape=(MAX_LEN,))
embedding_layer = Embedding(input_dim=VOCAB_SIZE, output_dim=embedding_dim, weights=[embedding_matrix], input_length=MAX_LEN, trainable=True)(input_layer)
x = Bidirectional(GRU(128, return_sequences=True, dropout=0.3, recurrent_dropout=0.3))(embedding_layer)
x = Bidirectional(GRU(64, dropout=0.3, recurrent_dropout=0.3))(x)
x = Dense(64, activation='relu')(x)
x = Dropout(0.3)(x)
output_layer = Dense(3, activation='softmax')(x)

# Define model
model = Model(inputs=input_layer, outputs=output_layer)

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
# Build LSTM-GRU model
model = Sequential([
    Embedding(input_dim=VOCAB_SIZE, output_dim=embedding_dim, weights=[embedding_matrix], trainable=True),
    Bidirectional(GRU(128, return_sequences=True, dropout=0.5, recurrent_dropout=0.5)),  # Keep return_sequences=True
    Bidirectional(GRU(64, return_sequences=False, dropout=0.5, recurrent_dropout=0.5)),  # Last layer should be False
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(3, activation='softmax')  # 3-class classification
])

In [ ]:
# Compile the model
model.compile(loss='sparse_categorical_crossentropy', optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4), metrics=['accuracy'])
#model.compile(loss='sparse_categorical_crossentropy', optimizer=Adam(learning_rate=1e-4), metrics=['accuracy'])

**Model Training**

In [ ]:
# Train model
#model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=15, batch_size=64)  # Increased batch size

In [ ]:
# Train the model
model.fit(X_train, y_train, validation_split=0.2, epochs=10, batch_size=32)

Epoch 1/20
293/293 ━━━━━━━━━━━━━━━━━━━━ 365s 1s/step - accuracy: 0.8307 - loss: 0.4336 - val_accuracy: 0.8165 - val_loss: 0.4870
Epoch 2/20
293/293 ━━━━━━━━━━━━━━━━━━━━ 377s 1s/step - accuracy: 0.8404 - loss: 0.4137 - val_accuracy: 0.8190 - val_loss: 0.4955
Epoch 3/20
293/293 ━━━━━━━━━━━━━━━━━━━━ 382s 1s/step - accuracy: 0.8390 - loss: 0.4243 - val_accuracy: 0.8182 - val_loss: 0.4984
Epoch 4/20
293/293 ━━━━━━━━━━━━━━━━━━━━ 374s 1s/step - accuracy: 0.8510 - loss: 0.3942 - val_accuracy: 0.8190 - val_loss: 0.4900
Epoch 5/20
293/293 ━━━━━━━━━━━━━━━━━━━━ 389s 1s/step - accuracy: 0.8469 - loss: 0.3992 - val_accuracy: 0.8182 - val_loss: 0.4954
Epoch 6/20
293/293 ━━━━━━━━━━━━━━━━━━━━ 358s 1s/step - accuracy: 0.8571 - loss: 0.3815 - val_accuracy: 0.8169 - val_loss: 0.4960
Epoch 7/20
293/293 ━━━━━━━━━━━━━━━━━━━━ 381s 1s/step - accuracy: 0.8595 - loss: 0.3656 - val_accuracy: 0.8190 - val_loss: 0.4865
Epoch 8/20
293/293 ━━━━━━━━━━━━━━━━━━━━ 383s 1s/step - accuracy: 0.8591 - loss: 0.3731 - val_accu

**Feature Extractions**

In [ ]:
# Ensure the model is called at least once before feature extraction
#_ = model.predict(np.zeros((1, X_train.shape[1])))  # Dummy prediction to initialize model

# Ensure the model is built by running a dummy input
#dummy_input = np.zeros((1, X_train.shape[1]))  # Shape must match input data
#model.predict(dummy_input)  # Run a dummy prediction to initialize model

# Extract features for RF
extractor = Model(inputs=model.input, outputs=model.layers[-3].output)  # Extract from the last hidden layer
#extractor = tf.keras.Model(inputs=model.input, outputs=model.layers[-2].output)  # Take features from last hidden layer
X_train_features = extractor.predict(X_train)
X_test_features = extractor.predict(X_test)

# Convert text to TF-IDF features for RF
vectorizer = TfidfVectorizer(max_features=5000)  # Limit TF-IDF features
X_train_tfidf = vectorizer.fit_transform(df['text']).toarray()

# Combine extracted features + TF-IDF
X_train_combined = np.hstack((X_train_features, X_train_tfidf[:len(X_train)]))
X_test_combined = np.hstack((X_test_features, X_train_tfidf[len(X_train):]))

366/366 ━━━━━━━━━━━━━━━━━━━━ 64s 152ms/step
92/92 ━━━━━━━━━━━━━━━━━━━━ 17s 189ms/step


In [ ]:
# Random Forest Model (Hyperparameter Tuning)
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

rf = RandomForestClassifier(random_state=42)
grid_search = RandomizedSearchCV(rf, param_grid, n_iter=10, cv=3, verbose=2, n_jobs=-1)
grid_search.fit(X_train_combined, y_train)

Fitting 3 folds for each of 10 candidates, totalling 30 fits


RandomizedSearchCV(cv=3, estimator=RandomForestClassifier(random_state=42),
                   n_jobs=-1,
                   param_distributions={'max_depth': [10, 20, None],
                                        'min_samples_leaf': [1, 2, 4],
                                        'min_samples_split': [2, 5, 10],
                                        'n_estimators': [100, 200, 300]},
                   verbose=2)

In [ ]:
# Best RF Model
best_rf = grid_search.best_estimator_
y_pred = best_rf.predict(X_test_combined)

# Evaluate Hybrid Model
accuracy = accuracy_score(y_test, y_pred)
print(f"🔥 Hybrid LSTM-GRU + RF Accuracy: {accuracy:.4f}")

🔥 Hybrid LSTM-GRU + RF Accuracy: 0.8224


In [ ]:
# Save model
model.save("hybrid_lstm_gru_rf_model.h5")

In [ ]:
def predict_sentiment(texts, model, tokenizer, label_encoder, max_length=100):
    if isinstance(texts, str):
        texts = [texts]  # Convert single text to a list

    # Preprocess texts
    sequences = pad_sequences(
        tokenizer.texts_to_sequences([clean_text(text) for text in texts]),
        maxlen=max_length,
        padding='post',
        truncating='post'
    )

    # Get model predictions
    probabilities = model.predict(sequences)  # Get confidence scores
    sentiment_labels = np.argmax(probabilities, axis=1)  # Get predicted class index

    # Convert predictions to human-readable labels
    predicted_sentiments = label_encoder.inverse_transform(sentiment_labels)

    # Combine labels with confidence scores
    results = list(zip(predicted_sentiments, probabilities.max(axis=1)))

    return results

# Load & Predict
loaded_model = tf.keras.models.load_model("hybrid_lstm_gru_rf_model.h5")

texts = ["I love flying with this airline!", "The worst experience ever!", "It was an okay flight."]
predictions = predict_sentiment(texts, loaded_model, tokenizer, label_encoder)

for text, (sentiment, confidence) in zip(texts, predictions):
    print(f"Text: {text}\nPredicted Sentiment: {sentiment} (Confidence: {confidence:.2f})\n")


1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
Text: I love flying with this airline!
Predicted Sentiment: positive (Confidence: 0.72)

Text: The worst experience ever!
Predicted Sentiment: negative (Confidence: 0.98)

Text: It was an okay flight.
Predicted Sentiment: neutral (Confidence: 0.67)

